# USD Swaps SDR Analytics (Package Detection)

This notebook demonstrates how to use **`USD_SOFR_SwapProduct`** in SDRUtils to:

1. Fetch USD rates swap trades from the DTCC SDR.
2. Classify trades using the USD SOFR swap product module.
3. Detect multi-leg packages (curve, fly, matched-maturity).
4. Summarize package legs into single rows for downstream analytics.


## Imports
Load SDRUtils components and pandas for data handling.

In [ ]:
from datetime import date, timedelta

import pandas as pd

from SDRUtils.data.builder import SDRDataBuilder
from SDRUtils.products import USD_SOFR_SwapProduct
from SDRUtils.products.usd.filters import new_sofr_swap_trades
from SDRUtils.registry import get_registry
from SDRUtils.core.classification import classifications_to_dataframe
from SDRUtils.packages import (
    detect_curve_trades_df,
    detect_fly_trades_df,
    detect_mms_trades_df,
    merge_package_legs_to_one_row,
)


## Configuration
Set a small date window to keep the SDR pull focused. Update as needed.

In [ ]:
end_date = date.today()
start_date = end_date - timedelta(days=5)

CACHE_PATH = "./cache/sdr"


## Fetch SDR Data
Pull DTCC SDR rates data for the selected window. This uses the cached builder.

In [ ]:
sdr = SDRDataBuilder(cache_path=CACHE_PATH, show_tqdm=True)

sdr_rates = sdr.fetch_historical_reports(
    agency="CFTC",
    asset_class="RATES",
    start_date=start_date,
    end_date=end_date,
    one_df=True,
)

sdr_rates.head()


## Filter USD SOFR Swaps
Use the SDRUtils USD filters to isolate SOFR OIS swap trades.

In [ ]:
sofr_trades = new_sofr_swap_trades(sdr_rates)

# Optional: keep a manageable sample size for quick iteration
sofr_trades = sofr_trades.head(2000).reset_index(drop=True)
sofr_trades["trade_id"] = range(1, len(sofr_trades) + 1)

sofr_trades.head()


## Classify Trades with USD_SOFR_SwapProduct
Instantiate the USD SOFR swap product module (via the registry if already registered)
and classify each trade to compute tenors, forwards, and trade labels.

In [ ]:
registry = get_registry()
product = registry.get_product("USD-SOFR-OIS") or USD_SOFR_SwapProduct()

classifications = [
    product.classify_trade(row, trade_id=int(row.trade_id), calculate_pv01=False)
    for _, row in sofr_trades.iterrows()
]

classified_df = classifications_to_dataframe(classifications)
classified_df.head()


## Join Classification + Raw SDR Fields
Package detection relies on additional SDR columns (underlier, platform, cleared flag).

In [ ]:
package_df = classified_df.merge(
    sofr_trades[[
        "trade_id",
        "UPI Underlier Name",
        "Platform identifier",
        "Cleared",
    ]],
    on="trade_id",
    how="left",
)

package_df.head()


## Detect Packages
Run curve, fly, and matched-maturity package detection sequentially.

In [ ]:
package_df = detect_curve_trades_df(package_df)
package_df = detect_fly_trades_df(package_df)
package_df = detect_mms_trades_df(package_df)

package_df["package_type"].value_counts()


## Package-Level Summary
Collapse package legs into a single row for analysis.

In [ ]:
package_summary = merge_package_legs_to_one_row(package_df)
package_summary.head()


## Next Steps
- Adjust the date range or sampling.
- Add curve-based PV01 by supplying a curve to `classify_trade`.
- Extend package analytics (e.g., spread metrics, roll tracking).
